# Validação do agente de reunião pt-BR (modelo treinado no Tinker)

Notebook enxuto para **carregar e validar** o modelo treinado pelo `meeting_agent_ptbr_colab.ipynb`, sem treinar nada.

**Como o carregamento funciona:** o `TinkerNativeBackend` guarda o estado do run (ids do Tinker + último step) em `.art/<project>/models/<name>/` na raiz do repositório. Registrar um `TrainableModel` com o **mesmo `name` e `project`** do treino reconecta no checkpoint mais recente — os pesos ficam na nuvem do Tinker.

**Requisitos:**
1. Rodar no **mesmo checkout/máquina** onde o treino aconteceu (o estado em `.art/` é local; para migrar de máquina, copie a pasta `.art/tag-ai-meeting-agent/`).
2. Chaves via **`.env` na raiz do repo** (copie o `.env.example`) ou secrets do Colab: `TINKER_API_KEY` (inferência) e `OPENAI_API_KEY` (juiz de correção). RULER/OpenRouter não são usados aqui.

**Custo:** inferência por token no Tinker (35B-A3B: US$ 0,36/M prefill + US$ 0,89/M sample) + poucas chamadas ao juiz.

Nota: além dos cenários `val` embutidos, o notebook carrega automaticamente o `scenarios_expanded.json` salvo pelo treino (se existir), incluindo o split `val` dele.

In [ ]:
# Mesma versão pinada do notebook de treino (bug do 0.5.18 fora de CUDA).
%pip uninstall -q -y openpipe-art
%pip install -q "openpipe-art[tinker] @ git+https://github.com/OpenPipe/ART.git@3492eb9a9ff1" openai langdetect python-dotenv

In [32]:
import os
from getpass import getpass

# Ordem de carregamento: 1) ambiente/.env  2) secrets do Colab  3) prompt.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))  # procura .env subindo até a raiz do repo
except ImportError:
    pass

def _secret(name, required=True):
    if not os.environ.get(name):
        try:
            from google.colab import userdata
            os.environ[name] = userdata.get(name) or ""
        except Exception:
            if required:
                os.environ[name] = getpass(f"{name}: ")
    return os.environ.get(name)

_secret("TINKER_API_KEY")
_secret("OPENAI_API_KEY")
assert os.environ.get("TINKER_API_KEY"), "TINKER_API_KEY é obrigatória (inferência)"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY é obrigatória (juiz de correção)"

import asyncio
import json
from openai import AsyncOpenAI

## Dataset (mesmos cenários do treino — usamos o split `val`)

In [33]:
from dataclasses import dataclass, field

@dataclass
class Segment:
    idx: int
    speaker: str
    text: str

@dataclass
class MeetingScenario:
    id: str
    title: str
    date: str
    segments: list
    question: str
    expected: str = ""
    split: str = "train"

def _seg(rows):
    return [Segment(i, sp, tx) for i, (sp, tx) in enumerate(rows)]

SCENARIOS = [
    MeetingScenario(
        id="planning-q3", title="Planejamento Q3 — Produto", date="2026-06-10",
        segments=_seg([
            ("Ana", "Bom dia pessoal. Hoje precisamos fechar o escopo do Q3."),
            ("Bruno", "A prioridade número um continua sendo o app de desktop, certo?"),
            ("Ana", "Sim. O desktop tem que sair até o fim de julho, é compromisso com o cliente."),
            ("Carla", "Eu assumo a parte de captura de áudio do desktop então."),
            ("Bruno", "E eu fico com a integração do Teams. Fecho a POC até dia 20."),
            ("Ana", "Perfeito. A Carla entrega captura até 25 de julho e o Bruno a POC do Teams até 20 de junho."),
            ("Carla", "Combinado. Alguma dependência externa que eu precise saber?"),
            ("Ana", "Só o SDK de captura, mas isso já está aprovado pelo jurídico."),
        ]),
        question="Quais são as tarefas atribuídas e seus responsáveis e prazos?",
        expected="Carla: captura de áudio do desktop até 25/07. Bruno: POC de integração do Teams até 20/06.",
        split="train",
    ),
    MeetingScenario(
        id="incident-review", title="Revisão de Incidente — Fila de Transcrição", date="2026-06-12",
        segments=_seg([
            ("Diego", "A fila de transcrição travou ontem às 14h por duas horas."),
            ("Elena", "A causa foi o pico de reuniões simultâneas, o worker não escalou."),
            ("Diego", "Precisamos de auto-scaling na fila até o fim da semana."),
            ("Elena", "Eu configuro o auto-scaling. Sexta no máximo."),
            ("Diego", "E vamos adicionar um alerta de profundidade de fila também."),
            ("Fábio", "Eu crio o alerta de fila, posso entregar na quinta."),
            ("Diego", "Ótimo. Nenhum dado de cliente foi perdido, só houve atraso."),
        ]),
        question="O que causou o incidente e quais ações foram definidas para evitar recorrência?",
        expected="Causa: pico de reuniões simultâneas sem auto-scaling do worker. Ações: Elena configura auto-scaling até sexta; Fábio cria alerta de profundidade de fila até quinta.",
        split="train",
    ),
    MeetingScenario(
        id="sales-sync", title="Sync Comercial — Conta Northwind", date="2026-06-15",
        segments=_seg([
            ("Gabriela", "A Northwind quer expandir de 50 para 300 licenças."),
            ("Hugo", "Ótimo. Eles pediram algum requisito novo?"),
            ("Gabriela", "Sim, isolamento de dados por região e retenção de 90 dias."),
            ("Hugo", "Retenção configurável já temos. Isolamento por região é roadmap."),
            ("Gabriela", "Eles topam esperar o isolamento se tivermos data marcada."),
            ("Hugo", "Vou levantar o esforço de isolamento por região com o time de dados."),
            ("Gabriela", "Fecho a proposta de 300 licenças até quarta que vem."),
        ]),
        question="Quais requisitos o cliente pediu e qual é o próximo passo de cada pessoa?",
        expected="Requisitos: isolamento de dados por região e retenção de 90 dias. Próximos passos: Hugo levanta o esforço de isolamento por região; Gabriela fecha a proposta de 300 licenças até quarta.",
        split="train",
    ),
    MeetingScenario(
        id="design-review", title="Design Review — Bot do Teams", date="2026-06-18",
        segments=_seg([
            ("Iara", "O @TagBot precisa responder menção em canal e em DM."),
            ("João", "Em canal, só responde se for explicitamente mencionado, senão vira spam."),
            ("Iara", "Concordo. E em DM responde sempre."),
            ("João", "A latência alvo é abaixo de 3 segundos para a primeira resposta."),
            ("Iara", "Eu escrevo a spec da máquina de estados do bot até segunda."),
            ("João", "Eu prototipo o handler de menção até quarta."),
        ]),
        question="Quais decisões de comportamento do bot foram tomadas e quem entrega o quê?",
        expected="Decisões: responde em canal só quando mencionado, sempre em DM, latência alvo <3s. Iara escreve a spec até segunda; João prototipa o handler de menção até quarta.",
        split="train",
    ),
    MeetingScenario(
        id="retention-policy", title="Política de Retenção e Consentimento", date="2026-06-20",
        segments=_seg([
            ("Lara", "Todo áudio gravado precisa de consentimento explícito do organizador."),
            ("Marcos", "E a retenção padrão do áudio bruto?"),
            ("Lara", "Trinta dias para o áudio, um ano para a transcrição."),
            ("Marcos", "Cada tenant pode encurtar, mas não estender além disso."),
            ("Lara", "Exato. Eu documento a política até sexta."),
            ("Marcos", "Eu implemento o job de expiração de áudio até o fim do mês."),
        ]),
        question="Quais são as regras de consentimento e retenção definidas?",
        expected="Consentimento explícito do organizador para gravar áudio. Retenção: 30 dias para áudio bruto, 1 ano para transcrição; tenant pode encurtar mas não estender. Lara documenta até sexta; Marcos implementa o job de expiração até o fim do mês.",
        split="train",
    ),
    MeetingScenario(
        id="hiring-plan", title="Plano de Contratação — Time de IA", date="2026-06-22",
        segments=_seg([
            ("Núbia", "Precisamos de dois engenheiros de ML no Q3."),
            ("Otávio", "Um sênior para o pipeline de RL e um pleno para avaliação."),
            ("Núbia", "Eu abro as duas vagas até quinta."),
            ("Otávio", "Eu preparo o desafio técnico de RL até a semana que vem."),
            ("Núbia", "Meta é fechar as contratações até o fim de agosto."),
        ]),
        question="Quantas vagas, com quais perfis, e quais são as ações e prazos?",
        expected="Duas vagas: um ML sênior (pipeline de RL) e um pleno (avaliação). Núbia abre as vagas até quinta; Otávio prepara o desafio técnico até a semana seguinte; meta de fechar até fim de agosto.",
        split="val",
    ),
    MeetingScenario(
        id="pricing-debate", title="Debate de Precificação", date="2026-06-24",
        segments=_seg([
            ("Paula", "O plano por assento não está funcionando para clientes grandes."),
            ("Rui", "Proponho cobrança híbrida: assento mais consumo de minutos transcritos."),
            ("Paula", "Risco de assustar o cliente com conta variável."),
            ("Rui", "Colocamos um teto mensal para dar previsibilidade."),
            ("Paula", "Gostei. Eu modelo três cenários de receita até terça."),
            ("Rui", "Eu levanto o custo real de transcrição por minuto até segunda."),
        ]),
        question="Qual mudança de precificação foi proposta e quais tarefas ficaram definidas?",
        expected="Proposta: cobrança híbrida (assento + consumo de minutos transcritos) com teto mensal. Paula modela três cenários de receita até terça; Rui levanta o custo real por minuto até segunda.",
        split="val",
    ),
    MeetingScenario(
        id="empty-decisions", title="Bate-papo sem decisões", date="2026-06-25",
        segments=_seg([
            ("Sofia", "Só queria alinhar como todos estão, sem pauta fechada hoje."),
            ("Tiago", "Por mim tudo tranquilo, semana calma."),
            ("Sofia", "Ótimo. Então não temos nada a decidir, bom descanso a todos."),
        ]),
        question="Quais tarefas e responsáveis foram definidos nesta reunião?",
        expected="Nenhuma tarefa foi definida; a reunião não teve decisões.",
        split="val",
    ),
]

def train_scenarios():
    return [s for s in SCENARIOS if s.split == "train"]

def val_scenarios():
    return [s for s in SCENARIOS if s.split == "val"]

print(f"{len(train_scenarios())} cenários de treino, {len(val_scenarios())} de validação")

5 cenários de treino, 3 de validação


In [34]:
# Carrega os cenários expandidos salvos pelo notebook de treino, se existirem.
EXPANDED_PATH = "scenarios_expanded.json"

if os.path.exists(EXPANDED_PATH):
    extra = [MeetingScenario(
        id=d["id"], title=d["title"], date=d["date"],
        segments=[Segment(**seg) for seg in d["segments"]],
        question=d["question"], expected=d.get("expected", ""),
        split=d.get("split", "train"),
    ) for d in json.load(open(EXPANDED_PATH))]
    SCENARIOS.extend(extra)
    print(f"+{len(extra)} cenários expandidos carregados -> {len(val_scenarios())} de validação")
else:
    print("scenarios_expanded.json não encontrado — validando só com os cenários base.")

+199 cenários expandidos carregados -> 23 de validação


## Ferramentas do agente e helpers (idêntico ao treino)

In [35]:
import re

TOOLS = [
    {"type": "function", "function": {
        "name": "search_transcript",
        "description": "Busca segmentos da transcrição que contenham as palavras-chave.",
        "parameters": {"type": "object", "properties": {
            "keywords": {"type": "array", "items": {"type": "string"}}},
            "required": ["keywords"]}}},
    {"type": "function", "function": {
        "name": "read_segment",
        "description": "Lê o texto completo de um segmento pelo seu índice.",
        "parameters": {"type": "object", "properties": {
            "segment_idx": {"type": "integer"}}, "required": ["segment_idx"]}}},
    {"type": "function", "function": {
        "name": "return_final_answer",
        "description": "Devolve a resposta final ao usuário e encerra a tarefa.",
        "parameters": {"type": "object", "properties": {
            "answer": {"type": "string"},
            "segment_refs": {"type": "array", "items": {"type": "integer"}}},
            "required": ["answer"]}}},
]

def _search(scenario, keywords):
    # O modelo às vezes manda keywords como string ("a, b, c") em vez de lista;
    # sem normalizar, o for itera caractere a caractere e casa com tudo.
    if isinstance(keywords, str):
        keywords = [k.strip() for k in keywords.split(",")]
    kws = [k.lower() for k in keywords if isinstance(k, str) and k.strip()]
    hits = [{"segment_idx": s.idx, "speaker": s.speaker, "snippet": s.text[:80]}
            for s in scenario.segments
            if any(k in s.text.lower() or k in s.speaker.lower() for k in kws)]
    return json.dumps({"results": hits[:10]}, ensure_ascii=False)

def _read(scenario, idx):
    for s in scenario.segments:
        if s.idx == idx:
            return json.dumps({"segment_idx": s.idx, "speaker": s.speaker, "text": s.text}, ensure_ascii=False)
    return json.dumps({"error": f"segmento {idx} não existe"}, ensure_ascii=False)

def _strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def _choice_as_context(choice):
    msg = choice.message
    out = {"role": "assistant", "content": _strip_think(msg.content or "")}
    if msg.tool_calls:
        out["tool_calls"] = [{"id": tc.id, "type": "function",
                              "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                             for tc in msg.tool_calls]
    return out

SYSTEM_PROMPT = (
    "Você é um assistente de reuniões em português do Brasil. Sua tarefa é "
    "responder à pergunta do usuário consultando a transcrição de uma reunião "
    "por meio das ferramentas disponíveis. Use search_transcript para "
    "localizar segmentos relevantes e read_segment para lê-los na íntegra. "
    "Responda SEMPRE em português, de forma objetiva e fiel à transcrição — "
    "não invente tarefas, decisões ou nomes que não estejam nos segmentos. "
    "Quando tiver a resposta, chame return_final_answer."
)

## Carregar o modelo treinado

O `register` lê o estado em `.art/` e reconecta no run do Tinker. O `assert` garante que não estamos validando o modelo stock por engano.

In [37]:
import art
from art.tinker_native import TinkerNativeBackend

BASE_MODEL = "Qwen/Qwen3.6-35B-A3B"

backend = TinkerNativeBackend()
model = art.TrainableModel(
    name="meeting-agent-ptbr-002",   # rodada real (ferramentas corrigidas); o
                                     # checkpoint antigo segue no name -001   # mesmo name/project do treino
    project="tag-ai-meeting-agent",
    base_model=BASE_MODEL,
)
await model.register(backend)

step = await model.get_step()
state = model.read_state() or {}
print(f"Step atual: {step} | runs Tinker: {state.get('tinker_run_ids')}")
assert step > 0, (
    "Step 0 = modelo stock. O estado do treino não foi encontrado em .art/ — "
    "rode este notebook no mesmo checkout onde o treino aconteceu."
)

Step atual: 10 | runs Tinker: ['7eb8042b-f215-5f5a-a520-a4aa0da60bfb:train:0', 'e0387b40-abc3-58f0-9226-15722794a7ec:train:0', '47bb3136-3814-580b-8fe6-8fcc6f93aed6:train:0', '6693fb8d-911e-53ce-b0bb-a3d332129186:train:0']


## Rollout (idêntico ao treino)

In [38]:
from art.trajectories import History

try:
    from langdetect import detect
except ImportError:
    detect = None

MAX_TURNS = 10

async def rollout(model, scenario):
    client = model.openai_client()
    # gpt-5.x na API da OpenAI rejeita max_tokens; modelos treináveis
    # (proxy do Tinker) usam max_tokens normalmente.
    token_kwargs = ({"max_tokens": 1024} if getattr(model, "trainable", False)
                    else {"max_completion_tokens": 1024})
    trajectory = art.Trajectory(
        messages_and_choices=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Reunião: {scenario.title} ({scenario.date}). "
                f"A transcrição tem {len(scenario.segments)} segmentos.\n\n"
                f"Pergunta: {scenario.question}")},
        ],
        metadata={"scenario_id": scenario.id, "split": scenario.split},
    )
    final_answer = None

    for _ in range(MAX_TURNS):
        completion = await client.chat.completions.create(
            # O nome de inferência é resolvido pelo backend (artifact do
            # W&B no serverless, alias name@step no Tinker/local) — nunca
            # passe model.name direto.
            model=model.get_inference_name(),
            messages=trajectory.messages(),
            tools=TOOLS, **token_kwargs,
        )
        choice = completion.choices[0]
        trajectory.messages_and_choices.append(choice)
        tool_calls = choice.message.tool_calls or []
        if not tool_calls:
            break
        for tc in tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            try:
                if tc.function.name == "search_transcript":
                    result = _search(scenario, args.get("keywords", []))
                elif tc.function.name == "read_segment":
                    idx = args.get("segment_idx", -1)
                    if isinstance(idx, list):  # o modelo às vezes manda lista
                        idx = idx[0] if idx else -1
                    result = _read(scenario, int(idx))
                elif tc.function.name == "return_final_answer":
                    final_answer = args.get("answer", "")
                    result = json.dumps({"ok": True}, ensure_ascii=False)
                else:
                    result = json.dumps({"error": "ferramenta desconhecida"}, ensure_ascii=False)
            except Exception as e:
                # Argumento malformado vira feedback para o modelo, nunca crash
                # do rollout (uma exceção derruba o gather do step inteiro).
                result = json.dumps({"error": f"argumentos inválidos: {e}"}, ensure_ascii=False)
            trajectory.messages_and_choices.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result})
        if final_answer is not None:
            break

    # Último recurso: estourou os turnos sem responder? Força uma resposta com
    # o que já foi lido — resposta parcial pontua; vazia é zero garantido.
    # (tool_choice é honrado pela API da OpenAI e ignorado sem erro pelo proxy
    # do Tinker; a instrução no user message cobre os dois casos.)
    if final_answer is None:
        trajectory.messages_and_choices.append({
            "role": "user",
            "content": ("Os turnos de busca acabaram. Responda AGORA chamando a "
                        "ferramenta return_final_answer com a melhor resposta "
                        "possível com base no que você já leu da transcrição."),
        })
        completion = await client.chat.completions.create(
            model=model.get_inference_name(),
            messages=trajectory.messages(),
            tools=TOOLS,
            tool_choice={"type": "function",
                         "function": {"name": "return_final_answer"}},
            **token_kwargs,
        )
        choice = completion.choices[0]
        trajectory.messages_and_choices.append(choice)
        for tc in choice.message.tool_calls or []:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if tc.function.name == "return_final_answer":
                final_answer = args.get("answer", "")
            trajectory.messages_and_choices.append(
                {"role": "tool", "tool_call_id": tc.id,
                 "content": json.dumps({"ok": True}, ensure_ascii=False)})

    trajectory.metrics["answered"] = 1.0 if final_answer else 0.0
    if detect and final_answer:
        try:
            trajectory.metrics["is_pt"] = 1.0 if detect(final_answer) == "pt" else 0.0
        except Exception:
            trajectory.metrics["is_pt"] = 0.0
    trajectory.metadata["final_answer"] = final_answer or ""
    return trajectory

# Workaround multi-turn Qwen 3.x: cada turno treina exatamente uma vez.
# IMPORTANTE: aplicar só DEPOIS do RULER — ele não aceita additional_histories
# (ValueError em art/rewards/ruler.py); o treino aceita.
def split_multiturn_for_training(trajectory):
    msgs = trajectory.messages_and_choices
    a_pos = [i for i, m in enumerate(msgs) if not isinstance(m, dict)]
    if len(a_pos) > 1:
        def _hist(pos):
            ctx = [m if isinstance(m, dict) else _choice_as_context(m) for m in msgs[:pos]]
            return ctx + [msgs[pos]]
        trajectory.additional_histories = [
            History(messages_and_choices=_hist(p)) for p in a_pos[1:]]
        trajectory.messages_and_choices = msgs[: a_pos[0] + 1]
    return trajectory

print("rollout definido")

rollout definido


## Juiz de correção e `validate` (idêntico ao treino)

In [39]:
judge_client = AsyncOpenAI()  # OPENAI_API_KEY
CORRECTNESS_JUDGE = "gpt-5.4-mini"

import re as _re

async def judge_correctness(scenario, answer):
    """Crédito parcial: fração (0-1) dos pontos da referência capturados."""
    if not scenario.expected:
        return 0.0
    prompt = (
        "Você avalia a resposta de um assistente de reuniões contra uma "
        "referência. Considere os pontos de informação da referência (tarefas, "
        "responsáveis, prazos, decisões, riscos). Calcule a fração deles "
        "corretamente capturada pela resposta, mesmo com outras palavras. "
        "Desconte por informações inventadas que contradigam a referência. "
        "Responda APENAS com um número inteiro de 0 a 100.\n\n"
        f"Pergunta: {scenario.question}\nReferência: {scenario.expected}\n"
        f"Resposta do agente: {answer}\n")
    resp = await judge_client.chat.completions.create(
        model=CORRECTNESS_JUDGE,
        messages=[{"role": "user", "content": prompt}],
        # gpt-5.x: max_tokens não é aceito; orçamento inclui raciocínio.
        max_completion_tokens=512)
    text = (resp.choices[0].message.content or "").strip()
    m = _re.search(r"\d{1,3}", text)
    return min(100, int(m.group())) / 100.0 if m else 0.0

async def validate(model):
    scenarios = val_scenarios()
    trajs = await asyncio.gather(*(rollout(model, s) for s in scenarios))
    scores = await asyncio.gather(*(
        judge_correctness(s, t.metadata.get("final_answer", ""))
        for s, t in zip(scenarios, trajs)))
    acc = sum(scores) / len(scores) if scores else 0.0
    ans = (sum(t.metrics.get("answered", 0.0) for t in trajs) / len(trajs)) if trajs else 0.0
    # pt medido só entre as respondidas — senão "não respondeu" contamina o idioma.
    pt_vals = [t.metrics["is_pt"] for t in trajs if "is_pt" in t.metrics]
    pt = (sum(pt_vals) / len(pt_vals)) if pt_vals else float("nan")
    print(f"  [validação] correção média (parcial): {acc:.2%} ({len(scores)} cenários) | "
          f"respondeu: {ans:.0%} | pt (entre respondidas): {pt:.0%}")
    return acc

## Rodar a validação

Protocolo fixo: **somente** os cenários `val` do `scenarios_expanded.json` + os 3 base — não gere cenários novos aqui, ou os números perdem comparabilidade entre rodadas. A métrica agora é **correção média com crédito parcial** (fração dos pontos da referência capturados) — não compare com os números do juiz binário de rodadas anteriores.

In [40]:
final = await validate(model)
print(f"\nCorreção no split de validação: {final:.2%}")

  [validação] correção: 42.86% (28 cenários) | respondeu: 100% | pt (entre respondidas): 100%

Correção no split de validação: 42.86%


## Diagnóstico por cenário

Com poucos cenários, o número agregado esconde o que importa: *qual* caso falhou e como. Esta célula mostra pergunta, resposta do agente, referência e o veredito do juiz, caso a caso.

In [41]:
scenarios = val_scenarios()
trajs = await asyncio.gather(*(rollout(model, s) for s in scenarios))
scores = await asyncio.gather(*(
    judge_correctness(s, t.metadata.get("final_answer", ""))
    for s, t in zip(scenarios, trajs)))

for s, t, ok in zip(scenarios, trajs, scores):
    n_tools = sum(1 for m in t.messages_and_choices
                  if isinstance(m, dict) and m.get("role") == "tool")
    print("=" * 72)
    print(f"[{'✅' if ok else '❌'}] {s.id} — {s.title}  (tool calls: {n_tools})")
    print(f"  Pergunta:   {s.question}")
    print(f"  Agente:     {t.metadata.get('final_answer', '(sem resposta)')[:400]}")
    print(f"  Referência: {s.expected[:400] or '(vazia — cenário de controle)'}")

[✅] hiring-plan — Plano de Contratação — Time de IA  (tool calls: 7)
  Pergunta:   Quantas vagas, com quais perfis, e quais são as ações e prazos?
  Agente:     De acordo com a transcrição:

*   **Quantas vagas:** 2 vagas.
*   **Perfis:**
    1.  1 Engenheiro de ML Sênior para o pipeline de RL.
    2.  1 Engenheiro de ML Pleno para avaliação.
*   **Ações e Prazos:**
    *   **Abertura das vagas:** Núbia abre as duas vagas até **quinta-feira**.
    *   **Preparo do Desafio:** Otávio prepara o desafio técnico de RL até **a semana que vem**.
    *   **Meta d
  Referência: Duas vagas: um ML sênior (pipeline de RL) e um pleno (avaliação). Núbia abre as vagas até quinta; Otávio prepara o desafio técnico até a semana seguinte; meta de fechar até fim de agosto.
[✅] pricing-debate — Debate de Precificação  (tool calls: 8)
  Pergunta:   Qual mudança de precificação foi proposta e quais tarefas ficaram definidas?
  Agente:     A mudança de precificação proposta foi cobrar um valor híbrido com bas

## Baseline stock no mesmo conjunto

O número do modelo treinado só faz sentido comparado ao **modelo sem treino nos mesmos cenários**. Um `TrainableModel` com nome novo não tem estado em `.art/` → step 0 → serve o modelo base puro.

In [42]:
baseline_model = art.TrainableModel(
    name="meeting-agent-ptbr-baseline",   # nome sem estado salvo => stock
    project="tag-ai-meeting-agent",
    base_model=BASE_MODEL,
)
await baseline_model.register(backend)
assert await baseline_model.get_step() == 0

print("Baseline (modelo stock, mesmos cenários):")
baseline = await validate(baseline_model)
print(f"\nDelta treinado vs stock: {final - baseline:+.2%}")

Baseline (modelo stock, mesmos cenários):
  [validação] correção: 35.71% (28 cenários) | respondeu: 96% | pt (entre respondidas): 100%

Delta treinado vs stock: +7.14%


## GPT-5.4-mini como agente (mesmo harness)

Roda o **mesmo rollout** — ferramentas, limite de turnos, juiz e cenários — trocando só o modelo: o `gpt-5.4-mini` da OpenAI no lugar do Qwen. Responde à pergunta "um modelo de prateleira apenas *prompted* é melhor ou pior que o nosso treinado?" — o número que justifica (ou não) o investimento em RL.

In [43]:
# art.Model (não treinável): mesmo harness, inferência na API da OpenAI.
gpt_model = art.Model(
    name="gpt-5.4-mini",
    project="tag-ai-meeting-agent",
    inference_api_key=os.environ["OPENAI_API_KEY"],
    inference_base_url="https://api.openai.com/v1/",
)

print("GPT-5.4-mini como agente:")
gpt_score = await validate(gpt_model)

print("\n=== Comparativo (mesmos cenários) ===")
print(f"Qwen treinado (step atual): {final:.2%}")
try:
    print(f"Qwen stock (baseline):      {baseline:.2%}")
except NameError:
    print("Qwen stock: rode a célula do baseline acima para o comparativo completo")
print(f"GPT-5.4-mini:               {gpt_score:.2%}")

GPT-5.4-mini como agente:
  [validação] correção: 42.86% (28 cenários) | respondeu: 100% | pt (entre respondidas): 100%

=== Comparativo (mesmos cenários) ===
Qwen treinado (step atual): 42.86%
Qwen stock (baseline):      35.71%
GPT-5.4-mini:               42.86%


## Pergunta ad-hoc

Teste o agente com qualquer pergunta sobre um dos cenários (ou monte um `Scenario` novo com a sua própria transcrição).

In [44]:
from dataclasses import replace

sc = replace(val_scenarios()[0], question="Quais tarefas foram atribuídas e a quem?")
traj = await rollout(model, sc)
print("Reunião: ", sc.title)
print("Pergunta:", sc.question)
print("Resposta:", traj.metadata["final_answer"])

Reunião:  Plano de Contratação — Time de IA
Pergunta: Quais tarefas foram atribuídas e a quem?
Resposta: As tarefas atribuídas foram:
- Núbia: abrir as duas vagas de engenheiro de ML até quinta-feira.
- Otávio: preparar o desafio técnico de RL até a próxima semana.
